# Bronze — Delta Lake particionado

Este notebook lê arquivos `ecommerce_rastreamento.parquet` da RAW, adiciona apenas auditoria da Bronze e salva como Delta no container `squad1`, particionado por ano, mês, dia e hora de ingestão.

In [0]:
%run ../utils/utils.ipynb

## Parâmetros da carga Bronze

In [0]:
# ==============================================================================
# PARAMETRIZAÇÃO GERAL
# ==============================================================================
import json
import uuid
from datetime import datetime, timezone

# Origem RAW
CONTAINER_ORIGEM = "raw"
PASTA_ORIGEM = "real-time-data"
NOME_ARQUIVO = "ecommerce_rastreamento.parquet"

# Destino físico
CONTAINER_DESTINO = "squad1"
CAMADA_DESTINO = "bronze"
ENTIDADE_DESTINO = "ecommerce_rastreamento"
PASTA_DESTINO = f"{CAMADA_DESTINO}/{ENTIDADE_DESTINO}"

# CORREÇÃO: Protocolo ABFSS Nativo do Spark
STORAGE_ACCOUNT = "internshipdatalake"
CAMINHO_DELTA_BRONZE = f"abfss://{CONTAINER_DESTINO}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{PASTA_DESTINO}"

# Controle de leitura em JSON
PASTA_CONTROLE_BASE = f"controle/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}"
DATA_EXECUCAO = datetime.now(timezone.utc)
ANO_CONTROLE = DATA_EXECUCAO.strftime("%Y")
MES_CONTROLE = DATA_EXECUCAO.strftime("%m")
DIA_CONTROLE = DATA_EXECUCAO.strftime("%d")
CAMINHO_CONTROLE_DIA = (
    f"{PASTA_CONTROLE_BASE}/{ANO_CONTROLE}/{MES_CONTROLE}/{DIA_CONTROLE}/controle_leitura.json"
)

FORCAR_REPROCESSAMENTO = False
SOBRESCREVER_BRONZE = False
RUN_ID = str(uuid.uuid4())
PARTICOES_BRONZE = []  # Sem particionamento para rastreamento

print("RUN_ID:", RUN_ID)
print("Destino Bronze:", CAMINHO_DELTA_BRONZE)



## INICIALIZAÇÃO DOS CLIENTS DO ADLS (SDK)

In [0]:
# ==============================================================================
# INICIALIZAÇÃO DOS CLIENTS DO ADLS (SDK)
# ==============================================================================
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# 1. Cria a credencial usando as variáveis que vieram do config.ipynb
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

# 2. Conecta no serviço geral do Data Lake
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)

# 3. Cria os "ponteiros" exatos que as suas funções precisam
file_system_raw = service_client.get_file_system_client(CONTAINER_ORIGEM)
container_squad1 = service_client.get_file_system_client(CONTAINER_DESTINO)

print("Conexões com os containers RAW e SQUAD1 estabelecidas com sucesso!")

## Garantir estrutura correta no container squad1

A entidade **não** deve ser criada na raiz do container. O destino correto é `squad1/bronze/ecommerce_rastreamento`.

In [0]:
# Criar apenas as pastas corretas
for pasta in [CAMADA_DESTINO, PASTA_DESTINO, PASTA_CONTROLE_BASE]:
    garantir_diretorio(container_squad1, pasta)

# Verificação de resíduo antigo na raiz
raiz = [item.name for item in container_squad1.get_paths(recursive=False)]

if ENTIDADE_DESTINO in raiz:
    print(f"ATENÇÃO: existe uma pasta antiga na raiz: {ENTIDADE_DESTINO}")
    print("Ela NÃO é usada por este notebook.")
    print(f"Para remover manualmente, execute: container_squad1.delete_directory('{ENTIDADE_DESTINO}')")
else:
    print(f"OK: não existe pasta incorreta {ENTIDADE_DESTINO} na raiz.")

## Localizar arquivos `ecommerce_rastreamento.parquet` na RAW


In [0]:
file_system_raw = service_client.get_file_system_client(file_system=CONTAINER_ORIGEM)

arquivos_encontrados = [
    p.name
    for p in file_system_raw.get_paths(path=PASTA_ORIGEM, recursive=True)
    if p.name.endswith(NOME_ARQUIVO)
]

print(f'Arquivos encontrados para {NOME_ARQUIVO}: {len(arquivos_encontrados)}')
for arquivo in arquivos_encontrados:
    print(arquivo)

if len(arquivos_encontrados) == 0:
    raise Exception(f'Nenhum arquivo encontrado para {NOME_ARQUIVO} em {CONTAINER_ORIGEM}/{PASTA_ORIGEM}')

## Localizar arquivos na RAW e aplicar controle de leitura


In [0]:
arquivos_encontrados = [
    p.name
    for p in file_system_raw.get_paths(path=PASTA_ORIGEM, recursive=True)
    if p.name.endswith(NOME_ARQUIVO)
]

# Ordena priorizando arquivos mais recentes
arquivos_encontrados = sorted(
    arquivos_encontrados,
    key=extrair_data_do_caminho_raw,
    reverse=True
)

arquivos_ja_lidos, arquivos_controle = carregar_arquivos_ja_lidos(
    container_client=container_squad1, 
    pasta_controle_base=PASTA_CONTROLE_BASE
)
delta_destino_valido = delta_existe(
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS
)

# Se o controle diz que leu tudo, mas o destino não é Delta válido, reprocessa tudo.
# Isso evita o caso: pasta física existe sem _delta_log ou Delta foi salvo no lugar errado.
if FORCAR_REPROCESSAMENTO or not delta_destino_valido:
    arquivos_para_processar = arquivos_encontrados
    if not delta_destino_valido:
        print("ATENÇÃO: destino Bronze ainda não é Delta válido. Reprocessando todos os arquivos para recriar o Delta.")
else:
    arquivos_para_processar = [
        arquivo for arquivo in arquivos_encontrados
        if arquivo not in arquivos_ja_lidos
    ]

print(f"Arquivos encontrados para {NOME_ARQUIVO}: {len(arquivos_encontrados)}")
print(f"Arquivos de controle encontrados: {len(arquivos_controle)}")
print(f"Arquivos já lidos no controle: {len(arquivos_ja_lidos)}")
print(f"Delta destino válido: {delta_destino_valido}")
print(f"Arquivos para processar nesta execução: {len(arquivos_para_processar)}")

for arquivo in arquivos_para_processar[:30]:
    print(arquivo)

if len(arquivos_para_processar) > 30:
    print(f"... mais {len(arquivos_para_processar) - 30} arquivos")



## Ler arquivos novos, unir em um único DataFrame e adicionar auditoria Bronze

In [0]:
if len(arquivos_para_processar) == 0:
    print("Nenhum arquivo novo para processar na Bronze.")
    TEM_ARQUIVO_NOVO = False
    df_bronze_micro_lote = None
else:
    TEM_ARQUIVO_NOVO = True
    pandas_dfs = []
    registros_controle_novos = []

    for arquivo in arquivos_para_processar:
        print(f"Lendo: {arquivo}")
        file_client = file_system_raw.get_file_client(arquivo)
        conteudo = file_client.download_file().readall()

        df_pd = pd.read_parquet(BytesIO(conteudo))

        # Auditoria Bronze: não transforma regra de negócio; apenas adiciona auditoria.
        df_pd["bronze_ingested_at"] = DATA_EXECUCAO.replace(tzinfo=None)
        df_pd["bronze_source_file"] = arquivo
        df_pd["bronze_run_id"] = RUN_ID
        df_pd["bronze_ingest_year"] = int(DATA_EXECUCAO.strftime("%Y"))
        df_pd["bronze_ingest_month"] = int(DATA_EXECUCAO.strftime("%m"))
        df_pd["bronze_ingest_day"] = int(DATA_EXECUCAO.strftime("%d"))
        df_pd["bronze_ingest_hour"] = int(DATA_EXECUCAO.strftime("%H"))

        pandas_dfs.append(df_pd)

        registros_controle_novos.append({
            "run_id": RUN_ID,
            "entidade": ENTIDADE_DESTINO,
            "arquivo_origem": arquivo,
            "data_leitura_utc": DATA_EXECUCAO.isoformat(),
            "camada_destino": CAMADA_DESTINO,
            "caminho_delta_destino": CAMINHO_DELTA_BRONZE,
            "qtd_registros_lidos": int(len(df_pd)),
            "status": "LIDO"
        })

    df_bronze_pd = pd.concat(pandas_dfs, ignore_index=True)
    df_bronze_micro_lote = spark.createDataFrame(df_bronze_pd)

    print("Total de registros no micro-lote Bronze:", df_bronze_micro_lote.count())
    display(df_bronze_micro_lote.limit(10))

##  Salvar Bronze como Delta físico e particionado por data/hora


In [0]:
if TEM_ARQUIVO_NOVO:
    modo_gravacao = "overwrite" if (FORCAR_REPROCESSAMENTO or SOBRESCREVER_BRONZE) else "append"
    print(f"Iniciando gravação na Bronze no modo: {modo_gravacao}")

    # Passamos o STORAGE_OPTIONS que foi criado no config.ipynb
    sucesso = gravar_delta(
        df=df_bronze_micro_lote,
        camada=CAMADA_DESTINO,
        tabela=ENTIDADE_DESTINO,
        storage_opts=STORAGE_OPTIONS,
        mode=modo_gravacao,
        particionar=False  # Mantemos False para pedidos
    )

    if sucesso:
        print("Processo de gravação finalizado perfeitamente via deltalake SDK!")
else:
    print("Gravação ignorada: não há arquivo novo.")

##  Atualizar controle de leitura somente após sucesso da gravação

In [0]:
if TEM_ARQUIVO_NOVO:
    controle_dia_existente = ler_json_adls(
        container_squad1,
        CAMINHO_CONTROLE_DIA,
        padrao=[]
    )

    # Evita duplicar o mesmo arquivo dentro do controle do dia
    arquivos_no_controle_dia = {
        item.get("arquivo_origem")
        for item in controle_dia_existente
    }

    registros_para_adicionar = [
        item for item in registros_controle_novos
        if item["arquivo_origem"] not in arquivos_no_controle_dia
    ]

    controle_atualizado = controle_dia_existente + registros_para_adicionar

    salvar_json_adls(
        container_squad1,
        CAMINHO_CONTROLE_DIA,
        controle_atualizado
    )

    print("Registros adicionados ao controle:", len(registros_para_adicionar))
else:
    print("Controle não atualizado: não houve gravação Bronze.")

##  Validação final

In [0]:
print("Arquivos físicos dentro do destino Bronze:")
try:
    for item in container_squad1.get_paths(path=PASTA_DESTINO, recursive=True):
        print(item.name)
except Exception as e:
    print("Não foi possível listar destino Bronze:", e)

print("\nValidação Delta:")
if delta_existe(camada=CAMADA_DESTINO, tabela=ENTIDADE_DESTINO, storage_opts=STORAGE_OPTIONS):
    print("Sucesso total! Tabela Delta validada e pronta para a Silver.")
else:
    print("Atenção: A tabela Delta não foi encontrada na validação.")

print("\nControle do dia:")
controle_validacao = ler_json_adls(container_squad1, CAMINHO_CONTROLE_DIA, padrao=[])
print("Registros no controle do dia:", len(controle_validacao))
for item in controle_validacao[:10]:
    print(item)